In [1]:
#Cell 1 — Install Dependencies
!pip install -q transformers trl peft accelerate bitsandbytes datasets wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [20]:
#Cell 2 — Login
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [9]:
#Cell 3 — Load Model in 4-bit (fits T4)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for training + attach LoRA adapters
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready! ✅")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713
Model ready! ✅


In [13]:
#Cell 4 — Define Reward Function
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        # completion is a list of dicts like [{"role": "assistant", "content": "..."}]
        output = completion[0]["content"] if isinstance(completion[0], dict) else completion[0]
        score = 0.0
        if len(output) > 100: score += 0.5
        if any(x in output for x in ["1.", "2.", "```", "•"]): score += 0.5
        if any(x in output.lower() for x in ["here", "you can", "try", "example"]): score += 0.3
        rewards.append(score)
    return rewards

In [14]:
#Cell 5 — GRPO Training
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

data = [
    {"prompt": "How do I create a Python function?"},
    {"prompt": "Explain what an API is."},
    {"prompt": "How do I use a for loop in Python?"},
    {"prompt": "What is machine learning?"},
    {"prompt": "How do I read a file in Python?"},
]
dataset = Dataset.from_list(data)

config = GRPOConfig(
    output_dir="./grpo-output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    logging_steps=1,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=reward_fn,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
print("Training done! ✅")

Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000


Training done! ✅
